# Reproducción de Gong et al. (2024): Circuit 6 con codificación tree-hybrid en MNIST

Objetivo: comprobar que el Circuit 6 (que corresponde al ansatz `unitary.U_6`, ver `src/qcnn_benchmark/models/qcnn_gong.py`) con codificación tree-structured hybrid amplitude (`BLOCK_SIZE=2`, "our encoding n=2" en la Tabla 1 del paper) se acerca a la exactitud publicada por Gong, Pei, Zhang y Zhou (2024), *"Quantum convolutional neural network based on variational quantum circuits"* (Optics Communications 550, 129993). El valor de referencia es 98.18% en MNIST (Tabla 1, fila Circuit 6, columna "Our encoding n=2", con PCA), usando el protocolo de entrenamiento de este benchmark.

Sobre el par de clases: el paper no indica qué par de dígitos usa en la Tabla 1. Se usa 0 vs 1, el mismo par de `00_reproduce_hur.ipynb`, porque es el más común en QCNN de 8 qubits sobre MNIST y es consistente con las exactitudes superiores a 97% de toda la tabla.

La lógica reutilizable (datos, muestreo, PCA, circuito, entrenamiento y métricas) está en `src/qcnn_benchmark/` y este notebook solo configura y ejecuta. Los detalles del circuito están en el docstring de `qcnn_gong.py`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

from qcnn_benchmark.data import load_mnist_pool
from qcnn_benchmark.representations import build_pca_dataset
from qcnn_benchmark.models import qcnn_gong
from qcnn_benchmark.training import train_binary_classifier, uniform_pi_init
from qcnn_benchmark.metrics import batch_accuracy, predict_labels

print("Parámetros entrenables (Circuit 6, 3 conv + 3 pool):", qcnn_gong.TOTAL_PARAMS, "(paper, Sec. 4.1: 10 por capa conv)")
print("Capacidad del encoding tree-hybrid (BLOCK_SIZE=2):", qcnn_gong.ENCODING_CAPACITY, "features")


## 1. Carga de datos y representación (PCA densa, 8 componentes, [0, π])

El muestreo estratificado y la PCA vienen de `qcnn_benchmark.representations.pca`, igual que en `00_reproduce_hur.ipynb` pero con `n_components=8`, porque la Tabla 1 de Gong et al. usa PCA-8. Dentro del adaptador las 8 features se rellenan con ceros hasta la capacidad de la codificación tree-hybrid, que es 12 con `BLOCK_SIZE=2`.

In [ ]:
X_ALL, Y_ALL = load_mnist_pool()
print("Pool MNIST combinado (train+test oficiales):", X_ALL.shape, Y_ALL.shape)

print("0 vs 1:")
rep_0v1 = build_pca_dataset(X_ALL, Y_ALL, class_pos=1, class_neg=0, n_components=8)


## 2. Entrenamiento

Protocolo de `qcnn_benchmark.training.train_binary_classifier`: BCE, Adam (lr=0.01, β1=0.9, β2=0.999), 200 actualizaciones, lote de 25, recorte de la norma global del gradiente a 5.0, early stopping con paciencia de 5 chequeos de validación (δ=1e-4, un chequeo cada 10 actualizaciones), inicialización uniforme en [-π, π] y selección del checkpoint con menor pérdida de validación.

Se usa una sola semilla (`RUN_SEED = 0`), igual que en los otros notebooks de reproducción, porque el objetivo es validar el adaptador contra el número publicado. Los experimentos, de E0A en adelante, usan 5 semillas y reportan media e intervalo de confianza.

In [ ]:
RUN_SEED = 0


def plot_loss_curve(result, title):
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(range(1, result["n_updates_run"] + 1), result["train_loss_history"],
            label="Pérdida de entrenamiento (por actualización)", alpha=0.7)
    val_updates, val_losses = zip(*result["val_loss_history"])
    ax.plot(val_updates, val_losses, "o-", label="Pérdida de validación (cada 10 actualizaciones)", color="darkorange")
    if result["stopped_early_at"]:
        ax.axvline(result["stopped_early_at"], color="red", linestyle="--", alpha=0.6, label="Early stopping")
    ax.set_xlabel("Actualización de parámetros")
    ax.set_ylabel("Entropía cruzada binaria")
    ax.set_title(title)
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()


def plot_confusion(y_true, y_pred, title, class_labels):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    fig, ax = plt.subplots(figsize=(4.5, 4))
    im = ax.imshow(cm, cmap="Blues")
    ax.set_xticks([0, 1])
    ax.set_xticklabels(class_labels)
    ax.set_yticks([0, 1])
    ax.set_yticklabels(class_labels)
    ax.set_xlabel("Predicción")
    ax.set_ylabel("Etiqueta real")
    ax.set_title(title)
    for i in range(2):
        for j in range(2):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center",
                    color="white" if cm[i, j] > cm.max() / 2 else "black")
    plt.colorbar(im, ax=ax, fraction=0.046)
    plt.tight_layout()
    plt.show()
    print(cm)
    return cm


def report_accuracy(train_acc, test_acc, tag, target_mean=None):
    print("=" * 70)
    print(f"{tag}, Circuit 6 + tree-hybrid encoding (Gong et al.), 1 semilla de ejecución")
    print("=" * 70)
    print(f"Exactitud de entrenamiento: {train_acc * 100:.2f}%")
    print(f"Exactitud de prueba:        {test_acc * 100:.2f}%")
    if target_mean is None:
        print("Sin objetivo publicado por Gong et al. para esta configuración.")
        return
    gap_pp = test_acc * 100 - target_mean
    print(f"Objetivo publicado (Tabla 1, Circuit 6, our encoding n=2, PCA): {target_mean}%")
    print(f"Diferencia (prueba - objetivo): {gap_pp:+.2f} puntos porcentuales")
    if abs(gap_pp) <= 1.0:
        print("-> Dentro de ~1pp del objetivo publicado: fidelidad razonable para 1 semilla.")
    elif abs(gap_pp) <= 3.0:
        print("-> Brecha moderada (1-3pp). Aceptable para 1 semilla vs. el promedio publicado.")
    else:
        print("-> Brecha grande (>3pp). Revisar antes de escalar a 5 semillas.")


## 3. Corrida: MNIST 0 vs 1 (referencia 98.18%)

In [ ]:
result_0v1 = train_binary_classifier(
    qcnn_gong.predict_proba, qcnn_gong.TOTAL_PARAMS, rep_0v1, uniform_pi_init,
    run_seed=RUN_SEED, tag="0v1",
)
plot_loss_curve(result_0v1, "MNIST 0 vs 1, Circuit 6 + tree-hybrid (Gong et al.), curva de pérdida")


In [ ]:
params_0v1 = result_0v1["params"]
train_acc_0v1 = batch_accuracy(qcnn_gong.predict_proba, params_0v1, rep_0v1["X_train"], rep_0v1["y_train"])
test_acc_0v1 = batch_accuracy(qcnn_gong.predict_proba, params_0v1, rep_0v1["X_test"], rep_0v1["y_test"])
report_accuracy(train_acc_0v1, test_acc_0v1, "MNIST 0 vs 1", target_mean=98.18)


In [ ]:
y_pred_test_0v1 = predict_labels(qcnn_gong.predict_proba, params_0v1, rep_0v1["X_test"])
_ = plot_confusion(rep_0v1["y_test"], y_pred_test_0v1,
                    "Matriz de confusión, MNIST 0 vs 1 (prueba)",
                    ["0 (dígito 0)", "1 (dígito 1)"])


## 4. Resumen

| Par de clases | Exactitud entrenamiento | Exactitud prueba | Objetivo Gong et al. | Comparable |
|---|---|---|---|---|
| 0 vs 1 | ver celda de resultados | ver celda de resultados | 98.18% (Tabla 1, Circuit 6, our encoding n=2) | Sí (par supuesto, ver Sec. 0) |

La comparación con 5 semillas se hace en `E0A_seleccion_candidato.ipynb`.